# Convolutional Neural Networks: Step by Step

We will be implementing the building blocks of a CNN!

- Convolution functions, including:
    - Zero Padding
    - Convolve window
    - Convolution forward
    - Convolution backward (optional)
- Pooling functions, including:
    - Pooling forward
    - Create mask
    - Distribute value
    - Pooling backward (optional)

This notebook will implement these functions from scratch in `numpy`. In the next notebook, you will use the TensorFlow equivalents of these functions to build the simple model:

<img src="images/model.png" style="width:800px;height:300px;">

## Import Library

In [6]:
import numpy as np
import matplotlib.pyplot as plt

## 1. CNN - FORWARD PASS

### 1.1 - Zero-Padding

**Zero-padding = adds 0 around the border of an image**:
<img src="images/PAD.png" style="width:600px;height:400px;">
<caption><center> <u> <font color='purple'> <b>Figure 1</b> </u><font color='purple'>  : <b>Zero-Padding</b><br> Image (3 channels, RGB) with a padding of 2. </center></caption>

The main benefits of padding are:

- It **allows** you to use a CONV layer **without necessarily shrinking the height and width of the volumes**. This is important for building deeper networks.

- It helps us **keep more of the information at the border** of an image. Without padding, very few values at the next layer would be affected by pixels at the edges of an image.

In [7]:
def zero_pad(X, pad):
    """
    Argument:
    X -- np.array of shape (m, n_H, n_W, n_C)

    Return:
    X_pad -- np.array of shape (m, n_H + 2 * pad, n_W + 2 * pad, n_C)
    """
    X_pad = np.pad(X, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode='constant', constant_values= (0, 0))
    return X_pad

### 1.2 - Single Step of Convolution

In this part, implement a single step of convolution, in which you **apply** the **filter to a single position of the input**. This will be used to build a convolutional unit, which:

- Takes an input volume
- Applies a filter at every position of the input
- Outputs another volume (usually of different size)

<img src="images/Convolution_schematic.gif" style="width:500px;height:300px;">
<caption><center> <u> <font color='purple'> <b>Figure 2</b> </u><font color='purple'>  : <b>Convolution operation</b><br> with a filter of 3x3 and a stride of 1 (stride = amount you move the window each time you slide) </center></caption>

In [8]:
def conv_single_step(a_slice_input_prev, W, b):
    """
    Apply one filter defined by parameters W on a single slice (a_slice_prev) of the output activation
    of the previous layer.

    Argument:
    a_slice_input_prev -- a slice of input data of shape (f, f, n_C_prev)
    W -- Weight parameters contained in a window - matrix of shape (f, f, n_C_prev)
    b -- Bias parameters contained in a window - matrix of shape (1, 1, 1)

    Return:
    Z -- a scalar value, the result of convolving the sliding window (W, b) on a slice x of the input data
    """
    s = a_slice_input_prev * W #element-wise
    Z = np.sum(s)
    Z += float(b)

    return Z

### 1.3 - CNN - Forward Pass

In the forward pass, you will **take many filters and convolve** them on the **input**. **Each 'convolution' gives you a 2D matrix output**. You will **then stack these outputs to get a 3D volume**:

<img src="images/conv_kiank.png" style="width:500px;height:300px;">
<caption><center> <u> <font color='purple'> <b>Figure 3</b> </u><font color='purple'>  : <b>Convolution on 3D volume</b><br> with a filter of fxf </center></caption>

**Reminder**:

The formulas relating the output shape of the convolution to the input shape are:

$$n_H = \Bigl\lfloor \frac{n_{H_{prev}} - f + 2 \times pad}{stride} \Bigr\rfloor +1$$
$$n_W = \Bigl\lfloor \frac{n_{W_{prev}} - f + 2 \times pad}{stride} \Bigr\rfloor +1$$
$$n_C = \text{number of filters used in the convolution}$$

To define **a_slice** you will need to first define its corners `vert_start`, `vert_end`, `horiz_start` and `horiz_end`. This figure may be helpful for you to find out how each of the corners can be defined using h, w, f and s in the code below.

<img src="images/vert_horiz_kiank.png" style="width:400px;height:300px;">
<caption><center> <u> <font color='purple'> <b>Figure 4</b> </u><font color='purple'>  : <b>Definition of a slice using vertical and horizontal start/end (with a 2x2 filter)</b> <br> This figure shows only a single channel.  </center></caption>

In [9]:
def conv_forward(A_prev, W, b, hparameters):
    """
    Implements the forward propagation for a convolution function

    Argument:
    A_prev -- output activations of the previous layer, numpy array of shape (m, n_H_prev, n_W_prev, n_C_prev)
    (Với layer đầu tiên (conv đầu): A_prev chính là input ảnh (X). Với các layer sau: A_prev là output (activation) của layer trước đó)

    W -- Weights/Filter window, numpy array of shape (f, f, n_C_prev, n_C)

    b -- Biases, numpy array of shape (1, 1, 1, n_C). Each filter has its own (single) bias.

    hparameters -- python dictionary containing "stride" and "pad"

    Returns:
    Z -- conv output, numpy array of shape (m, n_H, n_W, n_C)
    cache -- cache of values needed for the conv_backward() function
    """

    # 1. Get A_prev shape
    m, n_H_prev, n_W_prev, n_C_prev = A_prev.shape

    #2. Get W shape
    f, f, n_C_prev, n_C = W.shape

    #3. Get hparameters
    stride = hparameters["stride"]
    pad = hparameters["pad"]

    #4. Get CONV shape
    n_H = int((n_H_prev + 2 * pad - f)/stride) + 1
    n_W = int((n_W_prev + 2 * pad - f)/stride) + 1

    #5. Initialize output volume Z with zeros
    Z = np.zeros((m, n_H, n_W, n_C))

    #6. Create A_prev_pad by padding A_prev
    A_prev_pad = zero_pad(A_prev, pad)

    for i in range(m):
        for h in range(n_H):
            # Find the vertical start and end of the current "slice"
            vert_start = h * stride
            vert_end = vert_start + f

            for w in range(n_W):
                # Find the horizontal start and end of the current "slice"
                horiz_start = w * stride
                horiz_end = horiz_start + f

                for c in range(n_C):
                    # Use the corners to define the (3D) slice of a_prev_pad
                    a_slice_prev = A_prev_pad[i][vert_start:vert_end,horiz_start:horiz_end,:]

                    # Convolve the (3D) slice with the correct filter W and bias b, to get back one output neuron.
                    weights = W[:, :, :, c]
                    biases = b[:, :, :, c]
                    Z[i, h, w, c] = conv_single_step(a_slice_prev, weights, biases)

    # Save information in "cache" for the backprop
    cache = (A_prev, W, b, hparameters)

    return Z, cache

### 1.4 - Pooling Layer

The pooling (POOL) layer **reduces** the **height** and **width** of the **input**. It helps **reduce computation**, as well as helps make feature detectors more invariant to its position in the input. The two types of pooling layers are:

- **Max-pooling layer**: slides an ($f, f$) window over the input and **stores the max** value of the window in the output.

- **Average-pooling layer**: slides an ($f, f$) window over the input and **stores the average** value of the window in the output.

These pooling layers have **no parameters for backpropagation to train**, but they have **hyperparameters window size $f$**.

<table>
<td>
<img src="images/max_pool1.png" style="width:500px;height:300px;">
<td>

<td>
<img src="images/a_pool.png" style="width:500px;height:300px;">
<td>
</table>

</u><font color='purple'>
**Reminder**:
As there's no padding, the formulas binding the output shape of the pooling to the input shape is:

$$n_H = \Bigl\lfloor \frac{n_{H_{prev}} - f}{stride} \Bigr\rfloor +1$$

$$n_W = \Bigl\lfloor \frac{n_{W_{prev}} - f}{stride} \Bigr\rfloor +1$$

$$n_C = n_{C_{prev}}$$

In [10]:
def pool_forward(A_prev, hparameters, mode = "max"):
    """
    Implements the forward pass of the pooling layer

    Argument:
    A_prev -- output activations of the previous layer, numpy array of shape (m, n_H_prev, n_W_prev, n_C_prev)
    (Với layer đầu tiên (conv đầu): A_prev chính là input ảnh (X). Với các layer sau: A_prev là output (activation) của layer trước đó)

    hparameters -- python dictionary containing "f" and "stride"
    mode -- the pooling mode ("max" or "average")

    Returns:
    A -- output of pool layer, a numpy array shape (m, n_H, n_W, n_C)
    cache -- cache used in the backward pass of the pooling layer, contains the input and hparameters
    """

    # 1. Retrieve dimensions from the input shape
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape

    # 2. Retrieve hyperparameters from "hparameters"
    f = hparameters["f"]
    stride = hparameters["stride"]

    # 3. Define the dimensions of the output
    n_H = int(1 + (n_H_prev - f) / stride)
    n_W = int(1 + (n_W_prev - f) / stride)
    n_C = n_C_prev

    # 4. Initialize output matrix A
    A = np.zeros((m, n_H, n_W, n_C))

    for i in range(m):
        for h in range(n_H):
            # Find the vertical start and end of the current "slice"
            vert_start = h * stride
            vert_end = vert_start + f

            for w in range(n_W):
                # Find the horizontal start and end of the current "slice"
                horiz_start = w * stride
                horiz_end = horiz_start + f

                for c in range(n_C):
                    # Use the corners to define the current slice on the ith training example of A_prev, channel c.
                    a_prev_slice = A_prev[i][vert_start:vert_end, horiz_start:horiz_end, c]

                    # Compute the pooling operation on the slice.
                    if mode == "max":
                        A[i, h, w, c] = np.max(a_prev_slice)
                    elif mode == "average":
                        A[i, h, w, c] = np.mean(a_prev_slice)

    # 5. Store input and hparameters in "cache" for pool_backward()
    cache = (A_prev, hparameters)

    # 6. Making sure your output shape is correct
    assert(A.shape == (m, n_H, n_W, n_C))

    return A, cache

<font color='blue'>

**What you should remember**:

* A **convolution extracts features** from an input image by taking the **dot product** between the **input** data and a **3D array of weights (the filter)**.
* The **2D output of the convolution** is called the **feature map**
* A **convolution layer** is where the **filter slides over** the **image** and **computes dot product**
    * This transforms the input volume into an output volume of different size
* **Zero padding** helps **keep** more **information** at the **image borders**, and is helpful for **building deeper networks**, because you can build a CONV layer **without shrinking** the **height** and **width** of the volumes
* **Pooling layers** gradually **reduce** the **height** and **width** of the **input** by **sliding a 2D window** over each specified region, then **summarizing the features** in that region

## 2.CNN - BACKPROPAGATION

### 2.1 - CONV Backward Pass

We have 2 main layers: **Output Layer $L^{th}$** and **Hidden Layer $l^{th}$**. That means we have **2 ways** to compute **Gradient dA**.

- **Output Layer $L^{th}$:** need to "know the loss function".
    - $J = Loss(A^{[L]}, y)$
    - We compute **Directly Gradient**: $dA^{[L]} = \frac{\partial J}{\partial A^{[L]}}$

- **Hidden Layer $l^{th}:$** NO need to know loss function
    - We compute **Gradient**: $dA^{[l]} = W^{[l+1]} . dZ^{[l+1]}$

Other Gradients **dZ, dW, dB** are the **same** for both Output Layer and Hidden Layer, it has formula:
   - $dZ = dA . g'(Z)$ -- **_Gradient of pre-activation_** (w.r.t Loss: $dZ = \frac{\partial J}{\partial Z}$)
   - $dW = dZ . A^{T}_{prev}$ -- **_Gradient of weights_** (w.r.t Loss: $dW = \frac{\partial J}{\partial W}$)
   - $db = \sum dZ$ -- **_Gradient of bias_** (w.r.t Loss: $db = \frac{\partial J}{\partial b}$)

Below is detail process:

<img src="images/FORWARD - BACKWARD PROCESS.png" style="width:800px;height:300px;">

#### 2.1.1 - Computing dA:

This is the formula for computing $dA$ for a certain filter $W_c$:

$$dA \mathrel{+}= \sum _{h=0} ^{n_H} \sum_{w=0} ^{n_W} W_c \times dZ_{hw} \tag{1}$$

Where:
- $W_c$ -- filter
- $dZ_{hw}$ -- gradient of pre-activation at the $h^{th} row$ and $w^{th} column$ (scalar)

Note that at **each time**, you **multiply** the **same filter $W_c$** by a **different dZ** when updating dA. We do so mainly because when computing the forward propagation, each filter is dotted and summed by a different a_slice. Therefore, when computing the backprop for dA, you are just adding the gradients of all the a_slices.

#### 5.1.2 - Computing dW:
This is the formula for computing $dW_c$ ($dW_c$ is the derivative of one filter):

$$dW_c  \mathrel{+}= \sum _{h=0} ^{n_H} \sum_{w=0} ^ {n_W} a_{slice} \times dZ_{hw}  \tag{2}$$

Where:
- $a_{slice}$ -- a slice which was used to **generate the activation $Z_{ij}$**. Since it is the same $W$, we will just add up all such gradients to get $dW$.

#### 5.1.3 - Computing db:

This is the formula for computing $db$ for a certain filter $W_c$:

$$db = \sum_h \sum_w dZ_{hw} \tag{3}$$

Where:
- $db$ is computed by summing $dZ$.



In [ ]:
def conv_backward(dZ, cache):
    """
    Implement the backward propagation for a convolution function

    Arguments:
    dZ - gradient of pre-activation

    Return:
    """